In [1]:
import json
import pandas as pd
from collections import defaultdict

In [2]:
def load_jsonl(path):
    data = []
    try:
        with open(path, "r") as f:
            for line in f:
                data.append(json.loads(line))
    except:
        print(f"Warning: {path} not found, using dummy data")
        data = [
            {"text": "I love this", "true_labels": ["joy"], "pred_labels": ["joy"]},
            {"text": "This is bad", "true_labels": ["anger"], "pred_labels": ["sadness"]},
        ]
    return pd.DataFrame(data)

In [3]:
baseline_path = "outputs/baseline_preds.jsonl"
bert_path = "outputs/bert_preds.jsonl"

In [4]:
df_base = load_jsonl(baseline_path)
df_bert = load_jsonl(bert_path)

df_base.head()

,text,true_labels,pred_labels
0,I love this,[joy],[joy]
1,This is bad,[anger],[sadness]


In [5]:
def compute_metrics(y_true, y_pred):
    return {
        "micro_f1": 0.0,
        "macro_f1": 0.0,
        "precision": 0.0,
        "recall": 0.0
    }

In [6]:
metrics_base = compute_metrics(df_base["true_labels"], df_base["pred_labels"])
metrics_bert = compute_metrics(df_bert["true_labels"], df_bert["pred_labels"])

print("Baseline:", metrics_base)
print("BERT:", metrics_bert)

Baseline: {'micro_f1': 0.0, 'macro_f1': 0.0, 'precision': 0.0, 'recall': 0.0}
BERT: {'micro_f1': 0.0, 'macro_f1': 0.0, 'precision': 0.0, 'recall': 0.0}


In [7]:
def per_label_f1(df):
    stats = defaultdict(lambda: {"tp":0,"fp":0,"fn":0})

    for true, pred in zip(df["true_labels"], df["pred_labels"]):
        true, pred = set(true), set(pred)

        for l in true & pred:
            stats[l]["tp"] += 1
        for l in pred - true:
            stats[l]["fp"] += 1
        for l in true - pred:
            stats[l]["fn"] += 1

    rows = []
    for l, v in stats.items():
        p = v["tp"] / (v["tp"] + v["fp"] + 1e-8)
        r = v["tp"] / (v["tp"] + v["fn"] + 1e-8)
        f1 = 2*p*r/(p+r+1e-8)
        rows.append([l, f1])

    return pd.DataFrame(rows, columns=["label","f1"]).sort_values("f1", ascending=False)

In [8]:
f1_base = per_label_f1(df_base)

print("Top 5 easiest:")
print(f1_base.head())

print("\nTop 5 hardest:")
print(f1_base.tail())

Top 5 easiest:
     label   f1
0      joy  1.0
1  sadness  0.0
2    anger  0.0

Top 5 hardest:
     label   f1
0      joy  1.0
1  sadness  0.0
2    anger  0.0


In [9]:
errors = df_base[df_base["true_labels"] != df_base["pred_labels"]]

sample_errors = errors.head(5)

for _, row in sample_errors.iterrows():
    print("Text:", row["text"])
    print("True:", row["true_labels"])
    print("Pred:", row["pred_labels"])
    print("-"*50)

Text: This is bad
True: ['anger']
Pred: ['sadness']
--------------------------------------------------
